# Phân tích & Dự đoán Giá Xe Hai Bánh Điện (Xe đạp điện & Xe máy điện)

Notebook này hướng dẫn quy trình từ phân tích khám phá dữ liệu (EDA), trích xuất và chuẩn hóa đặc trưng, đến huấn luyện các mô hình học máy (Linear Regression, SVR, Random Forest, XGBoost, LightGBM) để dự báo giá xe hai bánh điện tại Việt Nam.

## 1. Thiết lập Môi trường & Khai báo Thư viện

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path("..")
sys.path.append(str(ROOT.resolve()))

DATA_DIR = ROOT / "data" / "interim"
PROCESSED_DIR = ROOT / "data" / "processed" / "two_wheelers"
REPORTS_DIR = ROOT / "reports" / "two_wheelers"

# Cài đặt chung cho đồ thị
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 11,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 120
})


## 2. Phân tích Khám phá Dữ liệu (EDA)
Tải tập dữ liệu đã gộp sạch `two_wheelers_eda_ready.csv` và kiểm tra các thống kê cơ bản.

In [ ]:
df = pd.read_csv(DATA_DIR / "two_wheelers_eda_ready.csv")
print(f"Tổng số bản ghi: {len(df):,}")
print(f"Các cột trong dữ liệu: {list(df.columns)}")
display(df.head(5))


In [ ]:
print("Tỷ lệ các loại xe điện:")
print(df['sub_type'].value_counts())

print("\nThống kê khuyết thiếu:")
print(df.isnull().sum())

print("\nprice_vnd Statistics")
print(df['price_vnd'].describe().apply(lambda x: format(x, 'f')))


### Trực quan hóa Phân phối Giá bán
So sánh giá của xe máy điện (`xe_may_dien`) và xe đạp điện (`xe_dap_dien`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Phân phối giá tổng quan
sns.histplot(data=df, x=df["price_vnd"] / 1e6, hue="sub_type", kde=True, bins=40, multiple="stack", ax=axes[0])
axes[0].set_xlabel("Giá xe (Triệu VND)")
axes[0].set_ylabel("Số lượng tin đăng")
axes[0].set_title("Biểu đồ phân phối Giá xe theo Loại")

# Hộp râu (Boxplot)
sns.boxplot(data=df, x="sub_type", y=df["price_vnd"] / 1e6, ax=axes[1])
axes[1].set_xlabel("Loại xe")
axes[1].set_ylabel("Giá xe (Triệu VND)")
axes[1].set_title("Hộp phân phối Giá xe")

plt.tight_layout()
plt.show()


### Hãng xe phổ biến nhất
Hiển thị Top các hãng xe hai bánh điện xuất hiện nhiều nhất.

In [ ]:
plt.figure(figsize=(10, 5))
brand_counts = df['brand'].value_counts().head(12)
sns.barplot(x=brand_counts.values, y=brand_counts.index, palette="viridis")
plt.xlabel("Số lượng tin đăng")
plt.ylabel("Hãng xe")
plt.title("Top 12 Hãng xe Hai Bánh Điện phổ biến nhất")
plt.show()


## 3. Chạy Quy trình Tạo Đặc trưng (Feature Engineering)
Tiến hành tiền xử lý, điền khuyết thiếu theo giá trị trung vị của tập Train (để tránh rò rỉ dữ liệu - target leakage), ánh xạ thông số kỹ thuật (Pin, Quãng đường, Công suất), mã hóa và phân tách dữ liệu thành tập Train/Test.

In [ ]:
# Chạy file xử lý đặc trưng
exec(open(ROOT / "src" / "features" / "build_features_twowheeler.py", encoding="utf-8").read())


## 4. Huấn luyện các mô hình học máy & Đánh giá (Model Benchmarking)
Chạy huấn luyện cho 5 thuật toán chính: **Linear Regression, SVR, Random Forest, XGBoost, và LightGBM**. Lưu kết quả đo lường độ chính xác.

In [ ]:
# Chạy file huấn luyện mô hình
exec(open(ROOT / "src" / "models" / "train_twowheeler.py", encoding="utf-8").read())


### Bảng so sánh kết quả các mô hình
Hiển thị chi tiết bảng kết quả để đánh giá mô hình tối ưu.

In [ ]:
metrics_df = pd.read_csv(REPORTS_DIR / "model_metrics.csv")
display(metrics_df.sort_values(by="Test R2", ascending=False))


### Biểu đồ đánh giá của Mô hình tốt nhất
Hiển thị biểu đồ Giá trị dự đoán vs Giá trị thực tế và phân phối sai số (residuals) của mô hình tối ưu.

In [ ]:
# Tải mô hình tốt nhất để vẽ hình trực quan ngay trong notebook
best_model_name = metrics_df.sort_values(by="Test R2", ascending=False).iloc[0]["Model"]
print(f"Mô hình tốt nhất được chọn: {best_model_name}")

# Tải dữ liệu scaled và unscaled đã lưu
X_train = pd.read_csv(PROCESSED_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DIR / "X_test.csv")
X_train_scaled = pd.read_csv(PROCESSED_DIR / "X_train_scaled.csv")
X_test_scaled = pd.read_csv(PROCESSED_DIR / "X_test_scaled.csv")
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv")["price_vnd"].values

# Tải lại kết quả đánh giá (chúng ta sẽ sinh lại hình trực tiếp)
import glob
from IPython.display import Image, display as ipy_display

# Đọc biểu đồ đã lưu dạng PDF
print("Các biểu đồ PDF đánh giá đã được tạo thành công trong thư mục: reports/two_wheelers/")
